# Обработка вакансий DevOps

**Домашнее задание 3 - вариант 11, DevOps-инженер**

Среда выполнения - Google Colab, CPU. Исходные файлы - `devops_code_complete.zip` и `it_vacancies_2024.csv` (или CSV.GZ) в `/content`

Этапы работы - запуск RustFS, загрузка CSV в бакет `hh-lake`, подсчёт частот навыков, пар и троек методами MapReduce и DuckDB, сверка результатов, анализ зарплат и трёх регионов, профилирование времени и памяти

Вычисления ДЗ3 используют RustFS в Google Colab. Локальный стенд macOS с EC 2+2 относится к ДЗ2

In [ ]:
import sys, subprocess, pathlib, json, platform, datetime, zipfile, os, time
ROOT = pathlib.Path('/content/devops_hw'); ROOT.mkdir(exist_ok=True)
archive = pathlib.Path('/content/devops_code_complete.zip')
if not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    archive = pathlib.Path(next(name for name in uploaded if name.endswith('.zip')))
with zipfile.ZipFile(archive) as z: z.extractall(ROOT)
installed = subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],capture_output=True,text=True)
if installed.returncode: print(installed.stdout,installed.stderr)
installed.check_returncode()
print('Рабочий каталог:', ROOT)

In [ ]:
def run_program(script, *arguments):
    logdir=ROOT/'results/logs'; logdir.mkdir(parents=True,exist_ok=True)
    name=pathlib.Path(script).stem+'_'+('_'.join(arguments) or 'run')
    process=subprocess.Popen([sys.executable,'-u',str(ROOT/script),*arguments],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (logdir/(name+'.log')).open('w') as log:
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line)
    if process.wait(): raise RuntimeError('Этап завершился с ошибкой: '+name)
def stage(name): run_program('src/pipeline.py',name)
run_program('src/bootstrap.py')

In [ ]:
import importlib.metadata as md, psutil, duckdb
checked=subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-v'],capture_output=True,text=True)
print(checked.stdout,checked.stderr); checked.check_returncode()
metadata={'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'platform':platform.platform(),'python':sys.version,'cpu_count':os.cpu_count(),'ram_gib':psutil.virtual_memory().total/1024**3,'packages':{p:md.version(p) for p in ['duckdb','boto3','psutil','reportlab','matplotlib']},'environment':'Google Colab CPU'}
(ROOT/'results').mkdir(exist_ok=True)
(ROOT/'results/environment.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
print(json.dumps(metadata,ensure_ascii=False,indent=2))


In [ ]:
source=pathlib.Path('/content/it_vacancies_2024.csv')
compressed=pathlib.Path('/content/it_vacancies_2024.csv.gz')
stage('upload')
stage('profile')
quality=json.loads((ROOT/'results/data_quality.json').read_text())
assert quality['duplicate_selected_ids']==0
assert max(map(int,quality['stack_distribution']))<=30
print('DevOps:',quality['selected_rows'],'из',quality['total_rows'])

## Контроль загрузки данных в RustFS

Проверяются наличие исходного CSV в бакете `hh-lake` через S3 HEAD, размер объекта, число записей и сохранённый журнал загрузки

In [ ]:
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(ROOT / 'src'))
import pipeline
s3 = pipeline.s3_client()
cfg = pipeline.config()
object_key = 'raw/it_vacancies_2024.csv'
metadata = s3.head_object(Bucket=cfg['bucket'], Key=object_key)
storage_audit = json.loads((ROOT / 'results/storage.json').read_text())
quality = json.loads((ROOT / 'results/data_quality.json').read_text())
assert metadata['ContentLength'] == 847619384
assert quality['total_rows'] == 229388
assert quality['selected_rows'] == 6503
display(pd.DataFrame([
    ('Хранилище', 'RustFS, Google Colab'),
    ('Бакет', cfg['bucket']),
    ('Объект', object_key),
    ('HEAD, размер файла', f"{metadata['ContentLength']:,}".replace(',', ' ') + ' байт'),
    ('Исходных вакансий', f"{quality['total_rows']:,}".replace(',', ' ')),
    ('DevOps-вакансий', f"{quality['selected_rows']:,}".replace(',', ' '))
], columns=['Проверка', 'Результат']))
print('Журнал загрузки и контроля S3')
print(json.dumps(storage_audit, ensure_ascii=False, indent=2))


In [ ]:
print((ROOT/'results/storage.json').read_text())
quality=json.loads((ROOT/'results/data_quality.json').read_text())
print('Rows:',quality['total_rows'],'DevOps:',quality['selected_rows'],'unique IDs:',quality['unique_vacancy_ids'])
print('Stack maximum:',max(map(int,quality['stack_distribution'])))
def stage(name):
    logdir=ROOT/'results/logs';logdir.mkdir(exist_ok=True)
    process=subprocess.Popen([sys.executable,'-u',str(ROOT/'src/pipeline.py'),name],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with (logdir/(name+'.log')).open('w') as log:
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line)
    if process.wait(): raise RuntimeError('Stage failed: '+name)
stage('benchmark')


In [ ]:
stage('analysis')


In [ ]:
stage('verify-analysis')
frequency_check=json.loads((ROOT/'results/verification.json').read_text())
salary_check=json.loads((ROOT/'results/analysis_verification.json').read_text())
assert frequency_check['all_exact'] and frequency_check['failure_exact']
assert salary_check['python_vs_duckdb_exact']
print('сверка пройдена: частоты, восстановление mapper, зарплаты и регионы')


In [ ]:
import pandas as pd
from IPython.display import display
for filename in ['benchmark_summary.csv','regions_summary.csv','salary_top.csv']:
    print('\n'+filename)
    display(pd.read_csv(ROOT/'results'/filename).head(32))
stage('publish')


In [ ]:
report_script=ROOT/'src/make_report.py'
report_script.write_text(report_script.read_text().replace('три крупнейших региона основной выборки','три крупных региона основной выборки'))
run_program('src/make_report.py')
from IPython.display import Image, FileLink
for chart in ['skills.png','performance.png','regions.png']:
    display(Image(filename=str(ROOT/'report'/chart),width=900))
pdf=ROOT/'report/ДЗ3_Зыков_Макар_DevOps.pdf'
assert pdf.exists() and pdf.stat().st_size>10000
print('PDF создан:',pdf, pdf.stat().st_size,'байт')
display(FileLink(str(pdf)))


In [ ]:
required=['storage.json','data_quality.json','benchmark_runs.csv','benchmark_summary.csv','verification.json','mapper_failure.json','analysis_verification.json','salary_all_supported.csv','salary_top.csv','regions_summary.csv','regions_skills_pairs_triples.csv','s3_objects.json']
for name in required:
    path=ROOT/'results'/name
    assert path.exists() and path.stat().st_size>0, name
assert (ROOT/'report/ДЗ3_Зыков_Макар_DevOps.pdf').exists()
sys.path.insert(0,str(ROOT/'src'))
import pipeline
s3=pipeline.s3_client(); cfg=pipeline.config()
s3.upload_file(str(ROOT/'report/ДЗ3_Зыков_Макар_DevOps.pdf'),cfg['bucket'],'report/devops_report.pdf')
manifest={'ready':True,'utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'checks':required,'github':'https://github.com/Sayflagdog/Data-Engineering-VK'}
(ROOT/'results/READY.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Проверены выходные файлы:', len(required))

In [ ]:
from google.colab import files
archive_path=pathlib.Path('/content/ДЗ3_Зыков_Макар_DevOps_results.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for directory in ['src','tests','sql','results','report']:
        for path in (ROOT/directory).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts:
                z.write(path,str(path.relative_to(ROOT)))
    for name in ['README.md','requirements.txt','.gitignore']:
        z.write(ROOT/name,name)
print('Архив:',archive_path,archive_path.stat().st_size,'байт')
files.download(str(archive_path))